## Example 1: Parsing numbers from different regions

In [ ]:
import phonenumbers

# Example 1: Basic phone number parsing
print("Example 1: Basic phone number parsing")

# Test numbers from different regions, with the country code and national number each should parse to
test_numbers = [
    ("+1 650-253-0000", "US", 1, 6502530000),
    ("+44 20 7946 0958", "GB", 44, 2079460958),
    ("+49 30 12345678", "DE", 49, 3012345678),
]

for number_str, region, expected_cc, expected_national in test_numbers:
    parsed = phonenumbers.parse(number_str, region)
    assert parsed.country_code == expected_cc, (number_str, parsed.country_code)
    assert parsed.national_number == expected_national, (number_str, parsed.national_number)
    assert phonenumbers.is_valid_number(parsed), number_str
    print(f"Input: '{number_str}'")
    print(f"  Country: {parsed.country_code}, National: {parsed.national_number}")
    print(f"  Valid: {phonenumbers.is_valid_number(parsed)}")
    print(f"  Type: {phonenumbers.number_type(parsed)}")

print("Example 1 completed")

## Example 2: Validating numbers

In [ ]:
import phonenumbers

# Example 2: Phone number validation
print("Example 2: Phone number validation")

# number -> (valid, possible), or the parse error it must raise
validation_tests = {
    "+1 650-253-0000": (True, True),
    "+44 20 7946 0958": (True, True),
    "+999 123 456 789": phonenumbers.NumberParseException.INVALID_COUNTRY_CODE,
    "+1 123": (False, False),
}

for test_num, expected in validation_tests.items():
    try:
        parsed = phonenumbers.parse(test_num, "US")
        is_valid = phonenumbers.is_valid_number(parsed)
        is_possible = phonenumbers.is_possible_number(parsed)
        
        print(f"Number: '{test_num}'")
        print(f"  Valid: {is_valid}, Possible: {is_possible}")
        assert (is_valid, is_possible) == expected, (test_num, is_valid, is_possible)
    except phonenumbers.NumberParseException as e:
        print(f"Number: '{test_num}' - Parse error: {e.error_type}")
        assert e.error_type == expected, (test_num, e.error_type)

print("Example 2 completed")

## Example 3: Formatting numbers

In [ ]:
import phonenumbers

# Example 3: Phone number formatting
print("Example 3: Phone number formatting")

sample_number = phonenumbers.parse("+1 650-253-0000", None)
print(f"Sample number: {sample_number}")

# Different format types
formats = [
    (phonenumbers.PhoneNumberFormat.E164, "E164"),
    (phonenumbers.PhoneNumberFormat.INTERNATIONAL, "International"),
    (phonenumbers.PhoneNumberFormat.NATIONAL, "National"),
]

expected_formats = {"E164": "+16502530000", "International": "+1 650-253-0000", "National": "(650) 253-0000"}
for format_type, format_name in formats:
    formatted = phonenumbers.format_number(sample_number, format_type)
    print(f"{format_name}: {formatted}")
    assert formatted == expected_formats[format_name], (format_name, formatted)

# Format for dialing from different regions
us_number = phonenumbers.parse("+1 650-253-0000", None)
regions = ["US", "GB", "DE"]

expected_dialing = {"US": "1 (650) 253-0000", "GB": "00 1 650-253-0000", "DE": "00 1 650-253-0000"}
for region in regions:
    dialing = phonenumbers.format_out_of_country_calling_number(us_number, region)
    print(f"Dial from {region}: {dialing}")
    assert dialing == expected_dialing[region], (region, dialing)

print("Example 3 completed")

## Example 4: Location and time zone lookup

In [ ]:
import phonenumbers
from phonenumbers import geocoder, timezone

# Example 4: Geographic information
print("Example 4: Geographic information")

# number -> (region code, time zone)
geo_test_numbers = {
    "+1 650-253-0000": ("US", "America/Los_Angeles"),
    "+44 20 7946 0958": ("GB", "Europe/London"),
    "+49 30 12345678": ("DE", "Europe/Berlin"),
    "+81 3-1234-5678": ("JP", "Asia/Tokyo"),
}

for number_str, (expected_region, expected_zone) in geo_test_numbers.items():
    parsed = phonenumbers.parse(number_str, None)
    geo = geocoder.description_for_number(parsed, "en")
    region = phonenumbers.region_code_for_number(parsed)
    zones = timezone.time_zones_for_number(parsed)
    
    print(f"Number: {number_str}")
    print(f"  Location: {geo}, Region: {region}")
    print(f"  Timezones: {list(zones)[:2]}")
    assert region == expected_region, (number_str, region)
    assert expected_zone in zones, (number_str, zones)
    assert isinstance(geo, str) and geo, (number_str, geo)

print("Example 4 completed")

## Example 5: Matching numbers written in different formats

In [ ]:
import phonenumbers

# Example 5: Number matching
print("Example 5: Number matching")

# Test if different formats represent same number; the national form is parsed in its own region
test_pairs = [
    ("+1 650-253-0000", "(650) 253-0000", "US", True),
    ("+44 20 7946 0958", "020 7946 0958", "GB", True),
    ("+1 650-253-0000", "+1 650-253-0001", "US", False),
]

for num1_str, num2_str, region, expected_same in test_pairs:
    num1 = phonenumbers.parse(num1_str, region)
    num2 = phonenumbers.parse(num2_str, region)
    
    are_same = num1 == num2
    match = phonenumbers.is_number_match(num1, num2)
    
    print(f"'{num1_str}' == '{num2_str}': {are_same}")
    assert are_same is expected_same, (num1_str, num2_str)
    expected_match = phonenumbers.MatchType.EXACT_MATCH if expected_same else phonenumbers.MatchType.NO_MATCH
    assert match == expected_match, (num1_str, num2_str, match)

# Format comparison
parsed = phonenumbers.parse("+1 650-253-0000", None)
print(f"\nE164: {phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164)}")
print(f"National: {phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.NATIONAL)}")
assert phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164) == "+16502530000"
assert phonenumbers.parse("+16502530000", None) == parsed  # E164 parses back to the same number

print("Example 5 completed")